In [1]:
import numpy as np
import pandas as pd
from typing import Tuple
from datetime import datetime, timedelta

# Pull Simulation

In [2]:
# Probability functions for character and lightcone banners (separate systems)

def character_probability(pity):
    """
    Return the 5-star character probability for the current pity.
    Base chance is 0.6% for pulls 0-72; soft pity increases linearly from pull 73 to 90;
    Hard pity at pull 90 yields a guaranteed 5-star.

    Args:
        pity (int): The current pity count (number of pulls since last 5-star).

    Returns:
        float: The probability of pulling a 5-star on the next pull.

    """
    if pity < 73:
        return 0.006
    elif pity < 90:
        # Linear increase from 0.006 at pity = 73 to ~1 at pity = 90.
        increment = 0.05847  # approximate increment per pull in soft pity range.
        return min(1.0, 0.006 + (pity - 73 + 1) * increment)
    else:
        return 1.0

def lightcone_probability(pity):
    """
    Return the 5-star Light Cone probability for the current LC pity.
    Base chance is 0.8% for pulls 0-64; soft pity increases linearly from pull 65 to 80;
    Hard pity at pull 80 is a guaranteed 5-star.

    Args:
        pity (int): The current LC pity count (number of pulls since last 5-star).

    Returns:
        float: The probability of pulling a 5-star on the next LC pull.
    """
    if pity < 65:
        return 0.008
    elif pity < 80:
        increment = 0.07  # approximate increment per pull in soft pity range.
        return min(1.0, 0.008 + (pity - 65 + 1) * increment)
    else:
        return 1.0

In [3]:
# Define pull simulation logic with the user's current situation

def simulate_combo(
    total_pulls: int,
    desired_chars: int,
    desired_lcs: int,
    start_char_pity: int = 0,
    start_char_guarantee: bool = False,
    start_lc_pity: int = 0,
    start_lc_guarantee: bool = False,
    full_4star_chars: bool = True,
    simulate_4star: bool = True
) -> Tuple[bool, int, float, float]:
    """
    Simulates pulls to reach desired characters and lightcones.
    Tracks refunded pulls and leftover pulls after simulation.

    Args:
        total_pulls (int): Total available pulls.
        desired_chars (int): Number of featured characters to obtain.
        desired_lcs (int): Number of featured lightcones to obtain.
        start_char_pity (int, optional): Pity count for the character banner. Defaults to 0.
        start_char_guarantee (bool, optional): Whether the next character is guaranteed. Defaults to False.
        start_lc_pity (int, optional): Pity count for the lightcone banner. Defaults to 0.
        start_lc_guarantee (bool, optional): Whether the next lightcone is guaranteed. Defaults to False.
        full_4star_chars (bool, optional): If True, refunds are granted for 4★ character duplicates. Defaults to True.
        simulate_4star (bool, optional): Whether to simulate 4★ pull refunds. Defaults to True.

    Returns:
        tuple[bool, int, int]:
            - success (bool): Whether all desired items were obtained.
            - pulls_used (int): Number of pulls used in the simulation.
            - leftover_pulls (int): Number of pulls remaining (including refunds).
    """

    # Track the number of pulls used, remaining, characters/ LCs obtained
    remaining = total_pulls
    used_pulls = 0
    refunded_pulls = 0.0
    chars_obtained = 0
    lcs_obtained = 0

    pity_char, guarantee_char = start_char_pity, start_char_guarantee
    pity_lc, guarantee_lc = start_lc_pity, start_lc_guarantee

    # Note: For now, we assume that users won't care to track when their last
    # 4-star was; this can be an optional param that is set later
    pity_4star_char = 0
    pity_4star_lc = 0

    # Pull so long as we have remaining pulls and we haven't received the number
    # of desired characters
    while remaining > 0 and (chars_obtained < desired_chars or lcs_obtained < desired_lcs):
        """
        Note: If the user wants to change strategy (LC first; one character copy
        first, then one LC, then try for another character; stop after loss),
        multiple calls to this simulation could be made based on outcomes
        """

        # Character banner pulls are prioritized first
        if chars_obtained < desired_chars:
            while remaining > 0 and chars_obtained < desired_chars:
                remaining -= 1
                used_pulls += 1

                if np.random.rand() < character_probability(pity_char):
                    if guarantee_char or np.random.rand() < 0.5:
                        chars_obtained += 1
                        guarantee_char = False
                    else:
                        guarantee_char = True
                    pity_char = 0
                    pity_4star_char = 0
                    break
                else:
                    pity_char += 1
                    pity_4star_char += 1

                    if simulate_4star and (pity_4star_char == 10 or np.random.rand() < 0.051):
                        pity_4star_char = 0
                        if full_4star_chars:
                            remaining += 1
                            refunded_pulls += 1.0

        # Light Cone banner pulls
        if lcs_obtained < desired_lcs and remaining > 0:
            while remaining > 0 and lcs_obtained < desired_lcs:
                remaining -= 1
                used_pulls += 1

                if np.random.rand() < lightcone_probability(pity_lc):
                    if guarantee_lc or np.random.rand() < 0.75:
                        lcs_obtained += 1
                        guarantee_lc = False
                    else:
                        guarantee_lc = True
                    pity_lc = 0
                    pity_4star_lc = 0
                    break
                else:
                    pity_lc += 1
                    pity_4star_lc += 1

                    if simulate_4star and (pity_4star_lc == 10 or np.random.rand() < 0.06):
                        pity_4star_lc = 0
                        remaining += 0.4
                        refunded_pulls += 0.4

    success = (chars_obtained >= desired_chars and lcs_obtained >= desired_lcs)
    pulls_leftover = remaining
    return success, used_pulls, round(refunded_pulls, 2), round(pulls_leftover, 2)

In [4]:
def run_simulation(
    total_pulls,
    desired_chars,
    desired_lcs,
    start_char_pity=0,
    start_char_guarantee=False,
    start_lc_pity=0,
    start_lc_guarantee=False,
    trials=10000
):

    """
    Runs a Monte Carlo simulation over multiple trials to estimate probability of
    success and average pulls used for obtaining the desired units.

    Args:
        inputs (BasicSimulationInput): Input dictionary with simulation configuration:
            - total_pulls
            - desired_chars
            - desired_lcs
            - start_char_pity
            - start_char_guarantee
            - start_lc_pity
            - start_lc_guarantee
            - trials

    Returns:
        tuple[float, float, float]:
            - Probability of success across trials.
            - Average number of pulls used.
            - Average number of pulls left over (including refunds).
    """

    successes = 0
    pulls_used = []
    pulls_refunded = []
    pulls_leftover = []

    # TODO: Log every run into a DF in case users want a view into that
    for _ in range(trials):
        success, used, refunded, leftover = simulate_combo(
            total_pulls,
            desired_chars,
            desired_lcs,
            start_char_pity,
            start_char_guarantee,
            start_lc_pity,
            start_lc_guarantee
        )
        if success:
            successes += 1
            pulls_used.append(used)
            pulls_refunded.append(refunded)
            pulls_leftover.append(leftover)

    avg_pulls = np.mean(pulls_used) if pulls_used else total_pulls
    avg_refunded = np.mean(pulls_refunded) if pulls_refunded else 0
    avg_leftover = np.mean(pulls_leftover) if pulls_leftover else 0
    probability = successes / trials
    return probability, avg_pulls, avg_refunded, avg_leftover

## Pull Simulation Test

In [5]:
scenarios = {
    "E1S1": {"desired_chars": 1, "desired_lcs": 1}
}

for name, params in scenarios.items():
    prob, avg_pulls, avg_refunded, avg_leftover = run_simulation(
        total_pulls=133,
        desired_chars=params["desired_chars"],
        desired_lcs=params["desired_lcs"],
        start_char_pity=0,
        start_char_guarantee=False,
        start_lc_pity=0,
        start_lc_guarantee=False,
        trials=50000
    )
    print(f"{name}: Probability={prob*100:.2f}%, Avg Pulls Used={avg_pulls:.2f}, Avg Pulls Refunded={avg_refunded:.2f}, Avg Pulls Leftover={avg_leftover:.2f}")

E1S1: Probability=40.97%, Avg Pulls Used=109.66, Avg Pulls Refunded=9.53, Avg Pulls Leftover=32.86


# Jades/ Pull Estimation

In [6]:
# Helper functions for regularly-occcurring events

def add_hoyolab_checkin(all_events, start_dt, end_dt):

    """
    Appends HoYoLAB login reward milestones (3x per month) to the event list.

    Args:
        all_events (list): The master list of (date, event name, jade) tuples.
        start_dt (datetime): Start date of the projection.
        end_dt (datetime): End date of the projection.

    Returns:
        None: Modifies all_events in place.
    """

    cur = datetime(start_dt.year, start_dt.month, 1)
    checkin_labels = {4: "1st", 12: "2nd", 19: "3rd"}

    while cur <= end_dt:
        year, month = cur.year, cur.month
        for day in [4, 12, 19]:
            try:
                checkin_date = datetime(year, month, day)
                if start_dt <= checkin_date <= end_dt:
                    label = checkin_labels.get(day, f"{day}th")
                    event_name = f"HoYoLAB Check-in ({label} Reward)"
                    all_events.append((checkin_date.strftime("%Y-%m-%d"), event_name, 20))
            except ValueError:
                continue  # skip invalid dates like Feb 30

        # move to next month
        if month == 12:
            cur = datetime(year + 1, 1, 1)
        else:
            cur = datetime(year, month + 1, 1)

def add_monthly_embers_reset(all_events, start_dt, end_dt):
    """
    Adds a row to the calendar for each 1st of the month to account for the
    5 free pulls from Embers exchange.

    Args:
        all_events (list): Event list to modify.
        start_dt (datetime): Projection start date.
        end_dt (datetime): Projection end date.
    """

    cur = datetime(start_dt.year, start_dt.month, 1)
    while cur <= end_dt:
        all_events.append((cur.strftime("%Y-%m-%d"), "Embers Exchange Reset", 0))
        next_month = cur.month % 12 + 1
        next_year = cur.year + (cur.month // 12)
        cur = datetime(next_year, next_month, 1)

def add_express_pass_renewal(all_events, express_start_str, end_dt):
    """
    Adds express supply renewal events every 30 days, each giving 300 jades.

    Args:
        all_events (list): Event list to modify.
        first_renewal_dt (datetime): Start date of the express pass renewal.
        end_dt (datetime): End date of the projection period.
    """

    er = datetime.strptime(express_start_str, "%Y-%m-%d")
    while er <= end_dt:
        all_events.append((er.strftime("%Y-%m-%d"), "Express Pass Renewal", 300))
        er += timedelta(days=30)

def add_endgame_resets(all_events, end_dt):
    """
    Adds recurring endgame content resets every 42 days (MoC, PF, Apoc Shadow), each giving 800 jades.

    Args:
        all_events (list): Event list to modify.
        end_dt (datetime): End date of projection.
    """

    def add_reset(start_str, name):
        d = datetime.strptime(start_str, "%Y-%m-%d")
        while d <= end_dt:
            all_events.append((d.strftime("%Y-%m-%d"), f"{name} Reset", 800))
            d += timedelta(days=42)
    add_reset("2025-03-03", "Apocalyptic Shadow")
    add_reset("2025-03-17", "Pure Fiction")
    add_reset("2025-03-31", "Memory of Chaos")

def add_weekly_simulated_universe(all_events, start_dt, end_dt):
    """
    Adds a weekly reward of 225 jades to simulate Simulated Universe weekly reset.

    Args:
        all_events (list): Event list to modify.
        start_dt (datetime): Start of projection.
        end_dt (datetime): End of projection.
    """

    cur = start_dt
    # Align to the next Monday if start_dt isn't a Monday
    while cur.weekday() != 0:  # 0 = Monday
        cur += timedelta(days=1)
    while cur <= end_dt:
        all_events.append((cur.strftime("%Y-%m-%d"), "Simulated Universe Weekly Reward", 225))
        cur += timedelta(weeks=1)

def add_nameless_honor_rewards(all_events, start_dt, end_dt):
    """
    Adds the full reward schedule of Nameless Honor (premium battle pass), assuming user purchases it.
    Includes immediate reward + estimated pull drops based on weekly progression.

    Args:
        all_events (list): The event list to append to.
        start_dt (datetime): Start date of projection.
        end_dt (datetime): End date of projection.
    """

    nh_start = datetime(2023, 4, 26)  # First Nameless Honor pass start
    cycle_length = timedelta(weeks=6)

    while nh_start <= end_dt:
        if nh_start >= start_dt:
            all_events.append((nh_start.strftime("%Y-%m-%d"), "Nameless Honor Purchase Bonus", 680))
            all_events.append(((nh_start + timedelta(days=7)).strftime("%Y-%m-%d"), "Nameless Honor Reward (1 Pull)", 160))
            all_events.append(((nh_start + timedelta(days=17)).strftime("%Y-%m-%d"), "Nameless Honor Reward (2 Pulls)", 320))
            all_events.append(((nh_start + timedelta(days=21)).strftime("%Y-%m-%d"), "Nameless Honor Reward (1 Pull)", 160))
        nh_start += cycle_length

In [7]:
def generate_pull_projection(
    current_pulls: int,
    start_date: str,
    end_date: str,
    events: list,
    express_renewal_start: str,
    nameless_honor: bool = False
) -> pd.DataFrame:
    """
    Generates a pull projection calendar showing when and how many pulls the user will accumulate
    based on known events, recurring income, and passive sources.

    Args:
        inputs (PullCalendarInput): Dictionary with projection parameters:
            - current_pulls (int): Starting pulls.
            - start_date (str): Start of the projection period (YYYY-MM-DD).
            - end_date (str): End of the projection period (YYYY-MM-DD).
            - events (list): List of tuples with (date, event name, jade amount).
            - express_renewal_start (str): First date of express pass renewal.
            - nameless_honor (bool): Whether Nameless Honor is purchased.

    Returns:
        pd.DataFrame: Table with columns [Date, Event, Jade Gain, Pull Gain, Cumulative Pulls].
    """

    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    jade_per_pull = 160.0
    daily_jade = 150  # 60 from daily missions + 90 from express supply pass


    # Build all events list
    all_events = list(events)

    # Add recurring events
    add_hoyolab_checkin(all_events, start_dt, end_dt)
    add_monthly_embers_reset(all_events, start_dt, end_dt)
    add_express_pass_renewal(all_events, express_renewal_start, end_dt)
    add_endgame_resets(all_events, end_dt)
    add_weekly_simulated_universe(all_events, start_dt, end_dt)
    if nameless_honor:
        add_nameless_honor_rewards(all_events, start_dt, end_dt)

    # Sort all events by date
    all_events.sort(key=lambda x: x[0])

    # Build the pull projection table
    cumulative = float(current_pulls)
    rows = []
    prev = start_dt

    for date_str, name, jade in all_events:
        d = datetime.strptime(date_str, "%Y-%m-%d")
        if d < prev:
            continue

        # Calculate passive daily income
        days = (d - prev).days
        passive_jade = days * daily_jade
        passive_pulls = passive_jade / jade_per_pull
        cumulative += passive_pulls

        # Event-specific pull gain
        if "Embers Exchange" in name:
            evt_pulls = 5
        elif "Express Pass Renewal" in name:
            evt_pulls = 300 / jade_per_pull
        elif jade == 0:
            evt_pulls = 0
        else:
            evt_pulls = jade / jade_per_pull

        cumulative += evt_pulls

        rows.append({
            "Date": d.date(),
            "Event": name,
            "Jade Gain": passive_jade + jade,
            "Pull Gain": round(passive_pulls + evt_pulls, 2),
            "Cumulative Pulls": int(cumulative)
        })

        prev = d

    # Final passive income to reach the exact end_date
    if prev < end_dt:
        days = (end_dt - prev).days
        passive_jade = days * daily_jade
        passive_pulls = passive_jade / jade_per_pull
        cumulative += passive_pulls
        rows.append({
            "Date": end_dt.date(),
            "Event": "End Date Passive Income",
            "Jade Gain": passive_jade,
            "Pull Gain": round(passive_pulls, 2),
            "Cumulative Pulls": int(cumulative)
        })

    df = pd.DataFrame(rows)
    return df

## Pull Simulation Test

In [8]:
events = [
    ("2025-04-09", "Star Rail WORLD Event Begins", 1200),  # Total from various objectives
    ("2025-04-11", "Seal Slammers Event Begins", 1320),   # Total from all stages and missions
    ("2025-04-26", "To The Ones That Blaze Anniversary Reward", 1600),  # 1600 Stellar Jades
    ("2025-04-26", "Astral Express Annual Trailblaze Report", 180),     # 180 Stellar Jades
    ("2025-04-28", "Morning Starlight Event Begins", 900),  # Estimated based on similar past events
    # No specific events with confirmed Stellar Jade rewards found for June 2025
]

In [9]:
pull_df = generate_pull_projection(current_pulls=93, start_date="2025-04-30", end_date="2025-06-11", events=events, express_renewal_start="2025-5-4", nameless_honor=True)
print(pull_df)

          Date                             Event  Jade Gain  Pull Gain  \
0   2025-05-01             Embers Exchange Reset        150       5.94   
1   2025-05-04     HoYoLAB Check-in (1st Reward)        470       2.94   
2   2025-05-04              Express Pass Renewal        300       1.88   
3   2025-05-05  Simulated Universe Weekly Reward        375       2.34   
4   2025-05-12     HoYoLAB Check-in (2nd Reward)       1070       6.69   
5   2025-05-12             Memory of Chaos Reset        800       5.00   
6   2025-05-12  Simulated Universe Weekly Reward        225       1.41   
7   2025-05-19     HoYoLAB Check-in (3rd Reward)       1070       6.69   
8   2025-05-19  Simulated Universe Weekly Reward        225       1.41   
9   2025-05-21     Nameless Honor Purchase Bonus        980       6.12   
10  2025-05-26          Apocalyptic Shadow Reset       1550       9.69   
11  2025-05-26  Simulated Universe Weekly Reward        225       1.41   
12  2025-05-28    Nameless Honor Rewar